# File 2 - Context Enrichment + Semantic Construction
### `02_context_semantic.ipynb`

Turns File 1's HR feature windows into **behavioral episodes** with semantic
context labels, then builds a typed **Behavioral Knowledge Graph (BKG)**.

**Pipeline (from the dev guide):**
1. Merge workouts + weather into every 15-min window (`is_workout` flag, interval-join workout HR)
2. Location clustering from workout GPS (DBSCAN, **haversine**)
3. Context Definitions: map `cluster -> location_label` (rule dict)
4. LLM **semantic normalization only** -> controlled vocab (Pydantic `Literal`, `temperature=0`)
5. Semantic Construction: build the BKG (nodes = episodes; temporal / similarity / context edges)

**Guardrails enforced here (violating = drifting from the design):**
- `merge_asof` always carries a `tolerance`; DBSCAN uses `metric='haversine'`.
- A DBSCAN cluster id is **not** a semantic label - labels come from `CONTEXT_RULES`.
- The LLM does **semantics only** - there is **no** `is_anomaly` field anywhere in File 2.
- tz-aware throughout (`Australia/Adelaide`); missing GPS -> `location_type='unknown'`, never a crash.

> **`TODO(user-input)`** markers flag the two specs the guide asks the user for
> (COPE ontology, Context Definitions). We ship documented defaults; swap them
> when the real specs arrive.

## 1. Imports & config

In [1]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

warnings.filterwarnings("ignore", category=FutureWarning)

TZ       = "Australia/Adelaide"
PROC_DIR = r"C:\Project\Apple Health Data\data\processed"
GPX_DIR  = r"C:\Project\Apple Health Data\data\apple_health_export\workout-routes"

os.makedirs(PROC_DIR, exist_ok=True)
print("Config OK ->", PROC_DIR)

Config OK -> C:\Project\Apple Health Data\data\processed


## 2. Load File 1 output contract

File 1 saved three parquet tables. If they are missing, run
`01_ingestion_signal_abstraction.ipynb` first (its final cell writes them).

In [2]:
def _load(name):
    p = os.path.join(PROC_DIR, name)
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"{name} not found in {PROC_DIR}.\n"
            "Run 01_ingestion_signal_abstraction.ipynb (final cell) to create the parquet files.")
    df = pd.read_parquet(p)
    # parquet round-trips tz, but be defensive about tz-awareness
    for col in ("datetime", "start_time", "end_time"):
        if col in df.columns and df[col].dt.tz is None:
            df[col] = df[col].dt.tz_localize(TZ)
    return df

df_hr_raw      = _load("hr_raw.parquet")
df_hr_features = _load("hr_features.parquet")
df_workouts    = _load("workouts.parquet")

print(f"df_hr_raw      : {len(df_hr_raw):,} rows")
print(f"df_hr_features : {len(df_hr_features):,} windows  cols={list(df_hr_features.columns)}")
print(f"df_workouts    : {len(df_workouts):,} workouts")
df_hr_features.head(3)

df_hr_raw      : 356,862 rows
df_hr_features : 104,565 windows  cols=['datetime', 'avg_hr', 'max_hr', 'min_hr', 'n_samples', 'hrv_sdnn']
df_workouts    : 289 workouts


,datetime,avg_hr,max_hr,min_hr,n_samples,hrv_sdnn
0,2017-11-27 10:45:00+10:30,82.000000,82.0,82.0,1,NaN
1,2017-11-27 11:00:00+10:30,73.500000,74.0,72.0,4,NaN
2,2017-11-27 11:15:00+10:30,76.833333,80.0,74.0,12,NaN


## Step 1 - Merge workouts & environment into every window

### 1a. `is_workout` flag + interval-joined workout HR
For each 15-min window we mark `is_workout=True` when it overlaps any workout
interval `[start_time, end_time]`. This flag drives REST-window filtering in
File 3 (the baseline must not mix circadian rhythm with exercise).

In [3]:
ep = df_hr_features.copy().sort_values("datetime").reset_index(drop=True)
ep["win_start"] = ep["datetime"]
ep["win_end"]   = ep["datetime"] + pd.Timedelta("15min")

# Vectorised overlap flag: a window is a workout window if it intersects any
# [start_time, end_time]. Workouts are few (289) so a simple interval sweep is fine.
ep["is_workout"] = False
w = df_workouts.dropna(subset=["start_time", "end_time"])
for s, e in zip(w["start_time"], w["end_time"]):
    ep.loc[(ep["win_start"] < e) & (ep["win_end"] > s), "is_workout"] = True

# Normalise Apple's raw workout activity string to a short activity hint.
# TODO(user-input): extend this map from the real Context Definitions taxonomy.
ACT_MAP = {
    "Running": "run", "Walking": "walk", "Cycling": "cycle", "Hiking": "walk",
    "Rowing": "row", "TraditionalStrengthTraining": "strength",
    "FunctionalStrengthTraining": "strength", "HighIntensityIntervalTraining": "run",
    "Elliptical": "run", "Swimming": "cycle", "CoreTraining": "strength",
    "Yoga": "rest", "Cooldown": "rest",
}
def act_hint(raw):
    if not isinstance(raw, str):
        return "unknown"
    key = raw.replace("HKWorkoutActivityType", "")
    return ACT_MAP.get(key, "unknown")

# Attach the dominant workout activity to each workout window (nearest by start).
w2 = w.assign(act=w["type"].map(act_hint)).sort_values("start_time")
ep["activity_hint"] = "rest"
if len(w2):
    idx = pd.merge_asof(
        ep[["win_start"]].reset_index(),
        w2[["start_time", "act"]].rename(columns={"start_time": "win_start"}),
        on="win_start", direction="nearest",
        tolerance=pd.Timedelta("15min"))            # guardrail: tolerance is mandatory
    ep.loc[ep["is_workout"], "activity_hint"] = (
        idx.set_index("index")["act"].reindex(ep.index).fillna("unknown")[ep["is_workout"]])
ep.loc[~ep["is_workout"], "activity_hint"] = "rest"

print(f"workout windows: {ep['is_workout'].sum():,} / {len(ep):,} "
      f"({ep['is_workout'].mean():.1%})")
print(ep["activity_hint"].value_counts())

workout windows: 921 / 104,565 (0.9%)
activity_hint
rest       103644
unknown       386
walk          342
row           170
cycle          17
run             6
Name: count, dtype: int64


### 1b. Parse workout GPS from GPX
GPS is only recorded **during workouts**, so most windows will have no location
(that is expected - they become `location_type='unknown'`). We parse the GPX
track points once and time-align them to windows.

In [4]:
def parse_gpx(path):
    """Return DataFrame [time(tz), lat, lon] for one GPX file."""
    try:
        root = ET.parse(path).getroot()
    except ET.ParseError:
        return None
    rows = []
    for pt in root.iter():
        if not pt.tag.endswith("trkpt"):
            continue
        lat, lon = pt.get("lat"), pt.get("lon")
        t = None
        for ch in pt:
            if ch.tag.endswith("time"):
                t = ch.text
        if lat and lon and t:
            rows.append((t, float(lat), float(lon)))
    if not rows:
        return None
    d = pd.DataFrame(rows, columns=["time", "lat", "lon"])
    d["time"] = pd.to_datetime(d["time"], utc=True).dt.tz_convert(TZ)   # GPX time is UTC 'Z'
    return d

gpx_files = glob.glob(os.path.join(GPX_DIR, "*.gpx"))
print(f"Found {len(gpx_files)} GPX files - parsing (subsample 1 point / 30s per file)...")

gps_parts = []
for f in gpx_files:
    d = parse_gpx(f)
    if d is None:
        continue
    d = d.set_index("time").resample("30s").first().dropna().reset_index()  # thin dense tracks
    gps_parts.append(d)

df_gps = (pd.concat(gps_parts, ignore_index=True).sort_values("time").reset_index(drop=True)
          if gps_parts else pd.DataFrame(columns=["time", "lat", "lon"]))
print(f"GPS points (thinned): {len(df_gps):,}")
df_gps.head(3)

Found 177 GPX files - parsing (subsample 1 point / 30s per file)...


GPS points (thinned): 10,269


,time,lat,lon
0,2018-01-07 14:09:30+10:30,-35.004800,138.597218
1,2018-01-07 14:10:00+10:30,-35.004879,138.597280
2,2018-01-07 14:10:30+10:30,-35.005022,138.597452


## Step 2 - Location clustering (DBSCAN, haversine)

`eps = 100 m / earth_radius` in radians; `metric='haversine'` on radians of
`(lat, lon)`. `-1` is noise. **The cluster id is NOT a semantic label** - Step 3
maps it to `home`/`work`/`gym` via rules.

In [5]:
from sklearn.cluster import DBSCAN

EARTH_M = 6_371_000.0
if len(df_gps):
    coords_rad = np.radians(df_gps[["lat", "lon"]].values)
    df_gps["cluster"] = DBSCAN(eps=100 / EARTH_M, min_samples=5,
                               metric="haversine").fit_predict(coords_rad)
    n_clusters = df_gps.loc[df_gps["cluster"] != -1, "cluster"].nunique()
    print(f"DBSCAN: {n_clusters} clusters, "
          f"{(df_gps['cluster'] == -1).mean():.1%} noise points")

    # cluster centroids (for weather coords + gym matching)
    cluster_centroids = (df_gps[df_gps["cluster"] != -1]
                         .groupby("cluster")[["lat", "lon"]].mean())
    print(cluster_centroids.round(4).to_string())
else:
    df_gps["cluster"] = pd.Series(dtype=int)
    cluster_centroids = pd.DataFrame(columns=["lat", "lon"])
    print("No GPS -> every window will be location_type='unknown' (expected, not an error).")

DBSCAN: 25 clusters, 0.3% noise points
             lat       lon
cluster                   
0       -35.0046  138.5976
1       -34.9239  138.6021
2       -34.9126  138.4910
3       -34.9317  138.6061
4        51.0203   -1.3140
5        51.4812   -0.4442
6        51.4985   -0.1797
7        51.5025   -0.1911
8        51.1562   -0.1646
9        59.4395   24.7558
10      -34.9287  138.6110
11      -34.9057  138.7083
12       59.3910   24.6705
13      -27.5546  153.0531
14      -27.5610  153.0687
15       59.3177   18.0527
16      -34.8938  138.6162
17      -35.5187  138.7638
18      -34.9047  138.5909
19      -34.9131  138.5982
20      -34.9806  138.5904
21       49.5089   11.7372
22       55.6854   12.5872
23       59.3325   18.0635
24       59.4417   24.7327


### 2b. Assign each window a location cluster (when GPS overlaps the window)
Only workout windows can get a cluster, since GPS exists only during workouts.
We take the modal cluster of GPS points falling inside each 15-min window.

In [6]:
ep["cluster"] = -1  # default: no location evidence
if len(df_gps) and (df_gps["cluster"] != -1).any():
    g = df_gps[df_gps["cluster"] != -1].copy()
    g["win_start"] = g["time"].dt.floor("15min")
    modal = (g.groupby("win_start")["cluster"]
             .agg(lambda s: s.value_counts().idxmax()))
    ep["cluster"] = ep["win_start"].map(modal).fillna(-1).astype(int)
print("windows with a location cluster:",
      int((ep["cluster"] != -1).sum()), "/", len(ep))

windows with a location cluster: 523 / 104565


## Step 1c - Weather enrichment (Open-Meteo archive, cached)

Open-Meteo's **archive** API is free and needs no key. We fetch hourly
`temperature_2m` and `relative_humidity_2m` once for the record's date span at a
representative coordinate (the densest GPS cluster, i.e. the likely home region;
falls back to Adelaide's centre if there is no GPS), **cache to parquet**, then
match onto windows by the **local wall-clock hour**.

> **DST-safe:** weather is fetched in **UTC** then converted to local (requesting
> local time trips the ambiguous autumn 02:00). Because that puts the hourly stamps
> at :30 local, the join floors **both** sides to the local hour (tz stripped) - an
> exact-hour merge would match nothing.
>
> Per-cluster weather is a minor refinement; since >99% of windows are
> `location='unknown'` a single representative coordinate is the honest default.
> `TODO(user-input)`: switch to per-cluster coords if higher spatial fidelity is needed.

In [7]:
WEATHER_CACHE = os.path.join(PROC_DIR, "weather_hourly.parquet")

# Representative coordinate: densest cluster centroid, else Adelaide CBD.
if len(cluster_centroids):
    densest = df_gps.loc[df_gps["cluster"] != -1, "cluster"].value_counts().idxmax()
    rep_lat, rep_lon = cluster_centroids.loc[densest, ["lat", "lon"]]
else:
    rep_lat, rep_lon = -34.9285, 138.6007   # Adelaide CBD fallback
print(f"weather coordinate: ({rep_lat:.4f}, {rep_lon:.4f})")

d0 = df_hr_features["datetime"].min().date()
d1 = df_hr_features["datetime"].max().date()

def fetch_weather():
    import requests
    # Fetch in UTC, then convert to local. Requesting local time (timezone=Adelaide)
    # and tz_localize-ing the result RAISES on the autumn DST fall-back (an ambiguous
    # 02:00). UTC has no DST, so parse-UTC -> tz_convert is safe and unambiguous.
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={rep_lat:.4f}&longitude={rep_lon:.4f}"
           f"&start_date={d0}&end_date={d1}"
           "&hourly=temperature_2m,relative_humidity_2m"
           "&timezone=UTC")
    r = requests.get(url, timeout=60); r.raise_for_status()
    h = r.json()["hourly"]
    wx = pd.DataFrame({
        "wx_hour": pd.to_datetime(h["time"]).tz_localize("UTC").tz_convert(TZ),
        "weather_temp": h["temperature_2m"],
        "weather_humidity": h["relative_humidity_2m"]})
    return wx

if os.path.exists(WEATHER_CACHE):
    df_weather = pd.read_parquet(WEATHER_CACHE)
    if df_weather["wx_hour"].dt.tz is None:
        df_weather["wx_hour"] = df_weather["wx_hour"].dt.tz_localize(TZ)
    print(f"weather loaded from cache: {len(df_weather):,} hours")
else:
    try:
        df_weather = fetch_weather()
        df_weather.to_parquet(WEATHER_CACHE, index=False)
        print(f"weather fetched & cached: {len(df_weather):,} hours")
    except Exception as e:                       # offline / API down -> degrade gracefully
        print(f"weather fetch failed ({e}); leaving weather columns NaN. TODO(user-input): retry online.")
        df_weather = pd.DataFrame(columns=["wx_hour", "weather_temp", "weather_humidity"])

weather coordinate: (-35.0046, 138.5976)
weather loaded from cache: 42,288 hours


In [8]:
# Match weather to each window on the LOCAL wall-clock hour. The weather timestamps
# sit at :30 (fetched in UTC then converted), so the old exact-hour merge on a :00
# floor matched NOTHING (0% coverage). Floor both sides to the local hour, with tz
# stripped so the autumn DST 02:00 does not raise, then map.
def _local_hour(s):
    naive = s.dt.tz_localize(None) if s.dt.tz is not None else s
    return naive.dt.floor("1h")

if len(df_weather):
    _wx = df_weather.dropna(subset=["wx_hour"]).copy()
    _wx["_h"] = _local_hour(_wx["wx_hour"])
    _wx = _wx.drop_duplicates("_h").set_index("_h")
    _eh = _local_hour(ep["datetime"])
    ep["weather_temp"] = _eh.map(_wx["weather_temp"]).to_numpy()
    ep["weather_humidity"] = _eh.map(_wx["weather_humidity"]).to_numpy()
else:
    ep["weather_temp"] = np.nan
    ep["weather_humidity"] = np.nan
print("weather coverage:", f"{ep['weather_temp'].notna().mean():.1%} of windows")
ep[["datetime", "avg_hr", "is_workout", "weather_temp", "weather_humidity"]].head(3)

weather coverage: 100.0% of windows


,datetime,avg_hr,is_workout,weather_temp,weather_humidity
0,2017-11-27 10:45:00+10:30,82.000000,False,20.2,48
1,2017-11-27 11:00:00+10:30,73.500000,False,22.8,43
2,2017-11-27 11:15:00+10:30,76.833333,False,22.8,43


## Step 3 - Context Definitions: cluster -> location label

**`TODO(user-input)`**: these are the guide's default heuristics, not the user's
real Context Definitions. `CONTEXT_RULES` is a plain config dict so it is trivial
to replace when the real taxonomy arrives.

Defaults:
- `home` = cluster with the most points in **00:00-05:00**
- `work` = cluster with the most points in **09:00-17:00 on weekdays**
- `gym`  = clusters that coincide with workout locations
- everything else / no GPS -> `unknown`

In [9]:
# TODO(user-input): replace with the real Context Definitions rule map / taxonomy.
CONTEXT_RULES = {
    "home_hours":  (0, 5),
    "work_hours":  (9, 17),
    "work_weekday_only": True,
}

cluster_label = {}   # cluster_id -> semantic label
if len(df_gps) and (df_gps["cluster"] != -1).any():
    g = df_gps[df_gps["cluster"] != -1].copy()
    g["hour"] = g["time"].dt.hour
    g["weekday"] = g["time"].dt.weekday < 5

    lo, hi = CONTEXT_RULES["home_hours"]
    night = g[g["hour"].between(lo, hi)]
    if len(night):
        cluster_label[int(night["cluster"].value_counts().idxmax())] = "home"

    lo, hi = CONTEXT_RULES["work_hours"]
    day = g[g["hour"].between(lo, hi)]
    if CONTEXT_RULES["work_weekday_only"]:
        day = day[day["weekday"]]
    if len(day):
        cand = int(day["cluster"].value_counts().idxmax())
        cluster_label.setdefault(cand, "work")

    # gym = clusters that appear during workouts (all GPS here is during workouts,
    # so any clustered location that is not already home/work is treated as gym/outdoor)
    for c in df_gps.loc[df_gps["cluster"] != -1, "cluster"].unique():
        cluster_label.setdefault(int(c), "outdoor")   # workout GPS outdoors by default

print("cluster -> label:", cluster_label or "(none - no GPS clusters)")

def loc_label(row):
    if row["cluster"] == -1:
        return "unknown"
    return cluster_label.get(int(row["cluster"]), "unknown")

ep["location_type"] = ep.apply(loc_label, axis=1)
print(ep["location_type"].value_counts())

cluster -> label: {9: 'home', 0: 'work', 1: 'outdoor', 2: 'outdoor', 3: 'outdoor', 4: 'outdoor', 5: 'outdoor', 6: 'outdoor', 7: 'outdoor', 8: 'outdoor', 10: 'outdoor', 11: 'outdoor', 12: 'outdoor', 13: 'outdoor', 14: 'outdoor', 15: 'outdoor', 16: 'outdoor', 17: 'outdoor', 18: 'outdoor', 19: 'outdoor', 20: 'outdoor', 21: 'outdoor', 22: 'outdoor', 23: 'outdoor', 24: 'outdoor'}


location_type
unknown    104042
work          283
outdoor       184
home           56
Name: count, dtype: int64


## Step 4 - LLM semantic normalization (controlled vocab only)

The LLM's **only** job is to map an ambiguous free-text context description onto
a controlled vocabulary. There is deliberately **no `is_anomaly` field** - the
LLM never judges anomalies (that would create circularity with the GNN).

To stay reproducible (`temperature=0`) and cheap, we LLM-classify the **distinct**
`(activity_hint, location_type, is_workout)` combinations - a handful of cases -
then map the labels back to all ~100k episodes.

Uses **Gemini** (`google-genai`, `gemini-3.1-flash-lite`) - the **same LLM stack as
File 3's baseline**, reading `GEMINI_API_KEY` from the project `.env`. If no key is
found it falls back to a deterministic rule map, so it always runs; the Pydantic
`Literal` schema validates either path.

In [10]:
from pydantic import BaseModel, ValidationError
from typing import Literal

class EpisodeContext(BaseModel):
    activity: Literal["rest", "walk", "run", "cycle", "row", "strength", "sleep", "unknown"]
    location_type: Literal["home", "work", "gym", "outdoor", "transit", "unknown"]
    # NOTE: intentionally NO is_anomaly / confidence field - LLM does semantics only.

# --- Gemini setup: SAME SDK / model / key as File 3's baseline (project consistency) ---
GEMINI_MODEL = "gemini-3.1-flash-lite"
_REPO_ROOT = os.path.dirname(os.path.dirname(PROC_DIR))          # ...\Apple Health Data
try:
    from dotenv import load_dotenv
    for _p in (os.path.join(_REPO_ROOT, ".env"),
               os.path.join(_REPO_ROOT, "notebooks", "enrichment_experiment", ".env")):
        if os.path.exists(_p):
            load_dotenv(_p, override=False)
except ImportError:
    pass
_GEMINI_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

# Distinct context cases to normalise (tiny set).
combos = (ep[["activity_hint", "location_type", "is_workout"]]
          .drop_duplicates().reset_index(drop=True))

def describe(row):
    hr_ctx = "during a workout" if row["is_workout"] else "at rest"
    return (f"A person is {hr_ctx}. Activity hint: '{row['activity_hint']}'. "
            f"Location: '{row['location_type']}'. Hour-of-day context: night if resting overnight.")

SYSTEM = ("You normalise wearable context descriptions to a controlled vocabulary. "
          "Return ONLY the activity and location_type enums. Never decide anomalies.")

def classify_llm(desc):
    """Gemini (google-genai), temperature=0 + structured output; None -> fallback.
    Same SDK / model / key as File 3 so the whole project uses one LLM stack."""
    if not _GEMINI_KEY:
        return None
    try:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=_GEMINI_KEY)
        cfg = types.GenerateContentConfig(
            temperature=0, response_mime_type="application/json",
            response_schema=EpisodeContext, system_instruction=SYSTEM,
            thinking_config=types.ThinkingConfig(thinking_budget=0))
        r = client.models.generate_content(model=GEMINI_MODEL, contents=desc, config=cfg)
        return r.parsed
    except Exception:                       # any failure -> deterministic fallback
        return None

def classify_fallback(row):
    """Deterministic controlled-vocab mapping used when no LLM is available."""
    act = row["activity_hint"] if row["activity_hint"] in EpisodeContext.model_fields[
        "activity"].annotation.__args__ else "unknown"
    loc = row["location_type"]
    return EpisodeContext(activity=act, location_type=loc)

USED_LLM = _GEMINI_KEY is not None
labels = []
for _, row in combos.iterrows():
    out = classify_llm(describe(row)) if USED_LLM else None
    if out is None:
        out = classify_fallback(row)
    labels.append((out.activity, out.location_type))
combos["activity"] = [a for a, _ in labels]
combos["loc_norm"] = [l for _, l in labels]

print("semantic normalization source:", f"Gemini {GEMINI_MODEL} (temperature=0)" if USED_LLM
      else "deterministic fallback (set GEMINI_API_KEY in .env to use the LLM)")
combos

semantic normalization source: Gemini gemini-3.1-flash-lite (temperature=0)


,activity_hint,location_type,is_workout,activity,loc_norm
0,rest,unknown,False,rest,unknown
1,walk,unknown,True,walk,unknown
2,walk,work,True,walk,work
3,unknown,work,True,unknown,work
4,row,unknown,True,row,unknown
5,walk,outdoor,True,walk,outdoor
6,unknown,unknown,True,unknown,unknown
7,unknown,outdoor,True,unknown,outdoor
8,row,outdoor,True,row,outdoor
9,walk,home,True,walk,home


In [11]:
# Map normalised labels back to every episode.
ep = ep.merge(combos[["activity_hint", "location_type", "is_workout", "activity", "loc_norm"]],
              on=["activity_hint", "location_type", "is_workout"], how="left")
ep["location_type"] = ep["loc_norm"].fillna(ep["location_type"])
ep = ep.drop(columns=["loc_norm"])

# Overnight rest -> 'sleep' (circadian context; still semantics, not anomaly).
_h = ep["datetime"].dt.hour
ep.loc[(~ep["is_workout"]) & (_h.between(0, 5)), "activity"] = "sleep"
ep["activity"] = ep["activity"].fillna("unknown")
print(ep["activity"].value_counts())

activity
rest        97572
sleep        6072
unknown       367
walk          342
row           170
strength       19
cycle          17
run             6
Name: count, dtype: int64


In [12]:
# --- Location context upgrade (shared module: notebooks/location_context) ---
# ADDITIVE enrichment: reverse-geocode the DBSCAN cluster centroids (cached) and
# upgrade location_type outdoor -> park/water/gym where the map says so, plus attach
# a real `location_place` name. The Step 3/4 home/work/outdoor labels are left
# untouched (geocoding a workout route cannot see where you live/work - that is a
# time-of-day signal). `park`/`water` extend the vocab post-LLM; they one-hot into
# the GCN node features automatically in File 3. Offline / no GPS -> no-op, never crashes.
import sys as _sys
_LC_DIR = os.path.join(_REPO_ROOT, "notebooks", "location_context")
if _LC_DIR not in _sys.path:
    _sys.path.insert(0, _LC_DIR)
import location_context as _lc

_loc_table = _lc.build_location_table(df_gps, online=True)   # geocode centroids (cached)
ep = _lc.attach_location(ep, df_gps, _loc_table)             # +location_place, upgrade outdoor->park/water/gym

# Persist the small artefacts context_providers.load_frames() reads for home_climate.
_lc.ensure_dirs()
if len(_loc_table):
    _loc_table.to_csv(os.path.join(_lc.RESULTS_DIR, "cluster_locations.csv"), index=False)
_home = _lc.derive_home_climate(_loc_table, df_weather)
with open(os.path.join(_lc.RESULTS_DIR, "home_climate.json"), "w", encoding="utf-8") as _f:
    json.dump(_home.__dict__, _f, indent=2, default=str)

print("location_type after geocode upgrade:")
print(ep["location_type"].value_counts())
print("episodes with a real place name:", ep["location_place"].notna().sum())
print("home_climate:", _home.place, "|", _home.band)


location_type after geocode upgrade:
location_type
unknown    104042
work          283
outdoor       156
home           56
water          16
park           12
Name: count, dtype: int64
episodes with a real place name: 523
home_climate: Gamma Crescent, Panorama | temperate


### Build `behavioral_episodes`
The episode table each downstream file consumes. Node ordering here **is** the
node index used by the graph in Step 5.

In [13]:
_ep_cols = [
    "datetime", "avg_hr", "max_hr", "min_hr", "hrv_sdnn", "n_samples",
    "is_workout", "activity", "location_type", "weather_temp", "weather_humidity",
]
if "location_place" in ep.columns:          # added by the location_context upgrade cell
    _ep_cols.append("location_place")
behavioral_episodes = ep[_ep_cols].copy().reset_index(drop=True)
behavioral_episodes.insert(0, "node_id", behavioral_episodes.index)
behavioral_episodes["timestamp_iso"] = behavioral_episodes["datetime"].apply(lambda t: t.isoformat())

EP_PATH = os.path.join(PROC_DIR, "behavioral_episodes.parquet")
behavioral_episodes.to_parquet(EP_PATH, index=False)
print(f"behavioral_episodes: {len(behavioral_episodes):,} rows -> {EP_PATH}")
behavioral_episodes.head(3)

behavioral_episodes: 104,565 rows -> C:\Project\Apple Health Data\data\processed\behavioral_episodes.parquet


,node_id,datetime,avg_hr,max_hr,min_hr,hrv_sdnn,n_samples,is_workout,activity,location_type,weather_temp,weather_humidity,location_place,timestamp_iso
0,0,2017-11-27 10:45:00+10:30,82.000000,82.0,82.0,NaN,1,False,rest,unknown,20.2,48,None,2017-11-27T10:45:00+10:30
1,1,2017-11-27 11:00:00+10:30,73.500000,74.0,72.0,NaN,4,False,rest,unknown,22.8,43,None,2017-11-27T11:00:00+10:30
2,2,2017-11-27 11:15:00+10:30,76.833333,80.0,74.0,NaN,12,False,rest,unknown,22.8,43,None,2017-11-27T11:15:00+10:30


## Step 5 - Semantic Construction: build the BKG

Nodes = episodes. Three edge types (guardrail: each edge carries an `edge_type`):
- **temporal** - consecutive episodes within a short gap (behavioural continuity)
- **similarity** - kNN in standardized feature space (kindred physiological states)
- **context** - episodes sharing `location_type` + `activity` within a time window

`TODO(user-input)`: `node_type`/`edge_type` names are placeholders until the COPE
ontology is provided. The graph is emitted as a PyG `Data` object plus flat
`node_table` / `edge_table` (kept for the File 4 audit).

In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

N = len(behavioral_episodes)

# --- node features (structural only; NO LLM anomaly labels) ---
t_hour = behavioral_episodes["datetime"].dt.hour + behavioral_episodes["datetime"].dt.minute / 60
feat = pd.DataFrame({
    "avg_hr":   behavioral_episodes["avg_hr"].fillna(behavioral_episodes["avg_hr"].median()),
    "max_hr":   behavioral_episodes["max_hr"].fillna(behavioral_episodes["max_hr"].median()),
    "min_hr":   behavioral_episodes["min_hr"].fillna(behavioral_episodes["min_hr"].median()),
    "hrv_sdnn": behavioral_episodes["hrv_sdnn"].fillna(behavioral_episodes["hrv_sdnn"].median()),
    "n_samples": behavioral_episodes["n_samples"],
    "hour_sin": np.sin(2 * np.pi * t_hour / 24),
    "hour_cos": np.cos(2 * np.pi * t_hour / 24),
    "is_workout": behavioral_episodes["is_workout"].astype(float),
})
X = StandardScaler().fit_transform(feat.values)
print("node feature matrix:", X.shape)

node feature matrix: (104565, 8)


In [15]:
# --- temporal edges: consecutive windows <= 30 min apart ---
dt = behavioral_episodes["datetime"].values
gap = pd.Series(behavioral_episodes["datetime"]).diff().dt.total_seconds().fillna(1e9).values
temporal_src, temporal_dst = [], []
for i in range(1, N):
    if gap[i] <= 30 * 60:           # 2 windows = 30 min
        temporal_src.append(i - 1); temporal_dst.append(i)
print(f"temporal edges: {len(temporal_src):,}")

# --- similarity edges: mutual kNN in feature space (k=8) ---
K = 8
nn = NearestNeighbors(n_neighbors=K + 1, algorithm="auto").fit(X)
_, nbr = nn.kneighbors(X)
sim_src, sim_dst = [], []
for i in range(N):
    for j in nbr[i, 1:]:            # skip self
        sim_src.append(i); sim_dst.append(int(j))
print(f"similarity edges: {len(sim_src):,}")

temporal edges: 102,644


similarity edges: 836,520


In [16]:
# --- context edges: same (location_type, activity), within +/- 2 hours ---
# Restrict to non-'unknown' context to keep this meaningful and bounded.
ctx_src, ctx_dst = [], []
ctx_df = behavioral_episodes[behavioral_episodes["location_type"] != "unknown"]
for (loc, act), grp in ctx_df.groupby(["location_type", "activity"]):
    idx = grp.index.values
    times = grp["datetime"].values
    for a in range(len(idx)):
        # link to a few temporally-near peers in the same context (cap fan-out)
        near = np.where(np.abs((times - times[a]).astype("timedelta64[s]").astype(float)) <= 2 * 3600)[0]
        for b in near[:5]:
            if idx[a] != idx[b]:
                ctx_src.append(int(idx[a])); ctx_dst.append(int(idx[b]))
print(f"context edges: {len(ctx_src):,}")

context edges: 848


In [17]:
# --- assemble edge table with types, then a PyG Data object ---
def _etable(src, dst, etype):
    return pd.DataFrame({"src": src, "dst": dst, "edge_type": etype})

edge_table = pd.concat([
    _etable(temporal_src, temporal_dst, "temporal"),
    _etable(sim_src, sim_dst, "similarity"),
    _etable(ctx_src, ctx_dst, "context"),
], ignore_index=True)

node_table = behavioral_episodes.copy()
node_table["node_type"] = "episode"     # TODO(user-input): refine via COPE ontology

node_table.to_parquet(os.path.join(PROC_DIR, "node_table.parquet"), index=False)
edge_table.to_parquet(os.path.join(PROC_DIR, "edge_table.parquet"), index=False)
np.save(os.path.join(PROC_DIR, "node_features.npy"), X)
print(f"node_table: {len(node_table):,}  edge_table: {len(edge_table):,} "
      f"({edge_table['edge_type'].value_counts().to_dict()})")

node_table: 104,565  edge_table: 940,012 ({'similarity': 836520, 'temporal': 102644, 'context': 848})


In [18]:
# PyG object (optional import - saved only if torch_geometric is available).
try:
    import torch
    from torch_geometric.data import Data

    edge_index = torch.tensor(edge_table[["src", "dst"]].values.T, dtype=torch.long)
    # integer code for edge_type so File 3 can weight reconstruction per relation
    etype_codes = {"temporal": 0, "similarity": 1, "context": 2}
    edge_type = torch.tensor(edge_table["edge_type"].map(etype_codes).values, dtype=torch.long)

    graph = Data(x=torch.tensor(X, dtype=torch.float),
                 edge_index=edge_index, edge_type=edge_type)
    graph.num_nodes = N
    torch.save(graph, os.path.join(PROC_DIR, "bkg_graph.pt"))
    print("saved PyG graph:", graph)
except ImportError:
    print("torch_geometric not installed - node_features.npy + edge_table.parquet are enough "
          "for File 3 to rebuild the graph. Install torch-geometric to persist bkg_graph.pt.")

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


saved PyG graph: Data(x=[104565, 8], edge_index=[2, 940012], edge_type=[940012], num_nodes=104565)


## File 2 output summary

Written to `data/processed/`:
- `behavioral_episodes.parquet` - the semantic episode table
- `node_table.parquet`, `edge_table.parquet` - flat BKG (for File 4 audit)
- `node_features.npy` - standardized node feature matrix
- `weather_hourly.parquet` - cached weather
- `bkg_graph.pt` - PyG graph (if torch_geometric available)

**Guardrail check:** merge_asof used `tolerance`; DBSCAN used haversine; cluster ids
were mapped to labels only via `CONTEXT_RULES`; the LLM produced **no** anomaly
field; missing GPS degraded to `unknown`; all timestamps are tz-aware.

Next: **File 3** consumes `behavioral_episodes` + the graph.